In [6]:
import python_obfuscator
from python_obfuscator.techniques import add_random_variables, one_liner, variable_renamer
import os
import shutil

In [5]:
PROJECT_EULER_PATH = "/home/danielrezende/datasets/PythonTheAlgorithms/project_euler"

# within each subfolder of PROJECT_EULER_PATH, pick the sol1.py, rename the file to subfolder_name.py, and copy to ./dataset/original
destination_path = "./dataset/original"
os.makedirs(destination_path, exist_ok=True)

for subfolder in os.listdir(PROJECT_EULER_PATH):
    subfolder_path = os.path.join(PROJECT_EULER_PATH, subfolder)
    if os.path.isdir(subfolder_path):
        source_file = os.path.join(subfolder_path, "sol1.py")
        if os.path.exists(source_file):
            destination_file = os.path.join(destination_path, f"{subfolder}.py")
            shutil.copy(source_file, destination_file)

In [9]:
# now obfuscate the code using python_obfuscator
def obfuscate(program: str, strategy: int):
    """
    Obfuscates a Python program.

    - program: the program to obfuscate, represented as a string
    - strategy: the obfuscation strategy to use (integer number from 0 to 7)
    """
    obfuscator = python_obfuscator.obfuscator()
    obfuscation_techniques = [
        [],  # 0 0 0
        [variable_renamer],
        [one_liner],
        [one_liner, variable_renamer],
        [add_random_variables],
        [add_random_variables, variable_renamer],
        [add_random_variables, one_liner],
        # [add_random_variables, one_liner, variable_renamer] | This won't do any obfuscation at all, and thus will be discarded
    ]
    obfuscated = obfuscator.obfuscate(
        program, remove_techniques=obfuscation_techniques[strategy]
    )
    if not isinstance(obfuscated, str):
        raise TypeError
    return obfuscated


# Generating all obfuscations for a given problem
# The strategies that passed the test in experiments.ipynb are deeemed safe,
# but we'll still have to double check the results anyway
SAFE_STRATEGIES = [2, 3, 6]


def generate_obfuscations(program: str)->list[str]:
    return [obfuscate(program, strategy) for strategy in SAFE_STRATEGIES]

In [14]:
# get the path of each file in ./dataset/original
original_files = [os.path.join(destination_path, file) for file in os.listdir(destination_path) if file.endswith('.py')]

# read each file, generate obfuscations, rename the new file to be {original_file.stem}_strat{i}.py, then write into ./dataset/obfuscated
obfuscated_path = "./dataset/obfuscated"
os.makedirs(obfuscated_path, exist_ok=True)

for original_file in original_files:
    with open(original_file, 'r') as f:
        program = f.read()
    obfuscations = generate_obfuscations(program)
    for i, obfuscated_program in enumerate(obfuscations):
        obfuscated_file = os.path.join(obfuscated_path, f"{os.path.splitext(os.path.basename(original_file))[0]}_strat_{i}.py")
        with open(obfuscated_file, 'w') as f:
            f.write(obfuscated_program)